In [55]:
import sys, os, glob, time, socket
from loguru import logger
from tabulate import tabulate
import pandas as pd
import numpy as np
import joblib   
import random

In [56]:
ANTIGENS = [
    "Diphtheria",
    "Pertussis",
    "Polio",
    "Tetanus",
    "Rotavirus",
    "PCV",
    "Measles",
    "Mumps",
    "Rubella",
    "Hepatitis_B",
    "Hib",
    "HPV",
]

N_YEARS = 10
YEARS = np.arange(1, N_YEARS + 1)

# Reformat the capacity scenarios

In [57]:
# capacity_scenario_names = [
#     "base_capacity",
#     "pandemic",
# ]
# capacity_scenario_probs = {
#     "base_capacity": 0.8418,
#     "pandemic": 0.0275,
# }

# capacity_scenario_names = [
#     "base_capacity",
#     "IPV_Shortage",
#     "pandemic",
#     "funding_delay",
#     "innacurate_forecast",
#     "supply_chain",
#     "other",
# ]
# capacity_scenario_probs = {
#     "base_capacity": 0.8418,
#     "IPV_Shortage": 0.0173,
#     "pandemic": 0.0275,
#     "funding_delay": 0.0778,
#     "innacurate_forecast": 0.0023,
#     "supply_chain": 0.0058,
#     "other": 0.0275,
# }

capacity_scenario_names = [
    "base_capacity"
]
capacity_scenario_probs = {
    "base_capacity": 1
}

In [58]:
dfs = []
for scenario in capacity_scenario_names:
    temp = pd.read_excel(
        f"data\production_capacity_scenarios_new_23JUN.xlsx",
        sheet_name=scenario,
    )
    temp["capacity_scenario"] = scenario
    temp["capacity_scenario_probability"] = capacity_scenario_probs[scenario]
    dfs.append(temp)

print(temp)
# Concatenate all the dataframes
capacity_scenarios = pd.concat(dfs, ignore_index=True)
# rename the columns
capacity_scenarios.rename(columns={"Manufacturer": "manufacturer"}, inplace=True)
# Convert years to columns
capacity_scenarios["capacity"] = capacity_scenarios[YEARS].values.tolist()
# Drop the years columns
capacity_scenarios.drop(YEARS, axis=1, inplace=True)

capacity_scenarios.to_csv("data\production_capacity_scenarios_base1.csv", index=False)

       Manufacturer          1  ...  capacity_scenario  capacity_scenario_probability
0       AJ_Vaccines    4458000  ...      base_capacity                              1
1          BB_NCIPD   31200000  ...      base_capacity                              1
2    Bharat_Biotech   45660000  ...      base_capacity                              1
3         Bilthoven    6960000  ...      base_capacity                              1
4      Biological_E  100200000  ...      base_capacity                              1
5    China_National    8340000  ...      base_capacity                              1
6               GSK  192600000  ...      base_capacity                              1
7      Haffkine_Bio   61200000  ...      base_capacity                              1
8           LG_Chem   47580000  ...      base_capacity                              1
9       Merck_Sharp   18590000  ...      base_capacity                              1
10           PT_Bio    8400000  ...      base_capacity

In [59]:
capacity_scenarios

,manufacturer,capacity_scenario,capacity_scenario_probability,capacity
0,AJ_Vaccines,base_capacity,1,"[4458000.0, 6120000.0, 7140000.0, 7140000.0, 7..."
1,BB_NCIPD,base_capacity,1,"[31200000.0, 34680000.0, 34920000.0, 33000000...."
2,Bharat_Biotech,base_capacity,1,"[45660000.0, 50520000.0, 51420000.0, 49800000...."
3,Bilthoven,base_capacity,1,"[6960000.0, 9540000.0, 11160000.0, 11100000.0,..."
4,Biological_E,base_capacity,1,"[100200000.0, 111600000.0, 112800000.0, 107400..."
5,China_National,base_capacity,1,"[8340000.0, 9120000.0, 9480000.0, 10200000.0, ..."
6,GSK,base_capacity,1,"[192600000.0, 202200000.0, 221400000.0, 224400..."
7,Haffkine_Bio,base_capacity,1,"[61200000.0, 67200000.0, 68400000.0, 66000000...."
8,LG_Chem,base_capacity,1,"[47580000.0, 54300000.0, 56400000.0, 53700000...."
9,Merck_Sharp,base_capacity,1,"[18590000.0, 18810000.0, 27500000.000000004, 7..."


In [60]:
import pandas as pd
import numpy as np

# Load data
demand_scenarios_df = pd.read_csv(
    "data/Medium_Demand_Structured_20_percent.csv",
    converters={"demand": pd.eval},
    usecols=["demand", "antigen", "demand_SID", "probability"]
)

# capacity_scenarios_df = pd.read_csv(
#     "data/production_capacity_scenarios.csv",
#     converters={"capacity": pd.eval},
# )

capacity_scenarios_df = capacity_scenarios

# Function to generate pairs
def generate_pairs(demand: pd.DataFrame, capacity: pd.DataFrame, n_pairs: int = 10, verbose: bool = True):
    demand_dict = demand.drop_duplicates(subset=["demand_SID"]).set_index("demand_SID")["probability"].to_dict()
    capacity_dict = capacity.drop_duplicates(subset=["capacity_scenario"]).set_index("capacity_scenario")["capacity_scenario_probability"].to_dict()

    demand_keys = list(demand_dict.keys())
    demand_probs = list(demand_dict.values())

    capacity_keys = list(capacity_dict.keys())
    capacity_probs = list(capacity_dict.values())

    if verbose:
        print(f"Unique demand scenarios: {demand_keys}")
        print(f"Unique capacity scenarios: {capacity_keys}")

    pairs = []
    pair_dfs = []
    prob_dict = {}
    pair_idx = 1

    for selected_capacity, selected_prob_capacity in capacity_dict.items():
        available_demand_keys = demand_keys.copy()
        available_demand_probs = demand_probs.copy()

        for _ in range(n_pairs):
            # Normalize the probabilities
            available_demand_probs = [p / sum(available_demand_probs) for p in available_demand_probs]

            selected_demand = np.random.choice(available_demand_keys, p=available_demand_probs)
            # Combine the probabilities
            selected_prob_demand = demand_dict[selected_demand]

            combined_prob = selected_prob_demand * selected_prob_capacity
            prob_dict[pair_idx] = combined_prob

            # Get the selected demand and capacity scenarios
            selected_demand_df = demand[demand["demand_SID"] == selected_demand].copy()
            selected_demand_df["type"] = "antigen"
            selected_demand_df.drop(columns=["probability", "demand_SID"], inplace=True)
            selected_demand_df.rename(columns={"antigen": "unit", "demand": "values"}, inplace=True)

            selected_capacity_df = capacity[capacity["capacity_scenario"] == selected_capacity].copy()
            selected_capacity_df["type"] = "manufacturer"
            selected_capacity_df.drop(columns=["capacity_scenario_probability", "capacity_scenario"], inplace=True)
            selected_capacity_df.rename(columns={"manufacturer": "unit", "capacity": "values"}, inplace=True)

            pair_df = pd.concat([selected_demand_df, selected_capacity_df])

            # Add additional information
            pair_df["pair_idx"] = pair_idx
            pair_df["pair"] = f"Demand: {selected_demand} - Capacity: {selected_capacity}"
            pair_dfs.append(pair_df)
            pairs.append((selected_demand, selected_capacity))
            pair_idx += 1

            # Remove the selected demand from the available demands
            index = available_demand_keys.index(selected_demand)
            available_demand_keys.pop(index)
            available_demand_probs.pop(index)

    pair_df = pd.concat(pair_dfs, ignore_index=True)

    # Calculate the sum of all probabilities
    total_sum = sum(prob_dict.values())

    # Scale the probabilities so they sum up to 1
    scaled_probabilities = {k: v / total_sum for k, v in prob_dict.items()}
    pair_df["pair_probability"] = pair_df["pair_idx"].map(scaled_probabilities)

    return pairs, pair_df

# Generate pairs
scenario_pairs, pair_df = generate_pairs(demand_scenarios_df, capacity_scenarios_df, n_pairs=1, verbose=True)

# Export to json and csv
pair_df.to_json("data/pair_demand_capacity_1scenario+20.json", orient="records", lines=True)
pair_df.to_csv("data/pair_demand_capacity_1scenario+20.csv", index=False)

# Display the DataFrame
pair_df


Unique demand scenarios: [20]
Unique capacity scenarios: ['base_capacity']


,unit,values,type,pair_idx,pair,pair_probability
0,Measles,"[504116405.1472744, 441959452.05863553, 486590...",antigen,1,Demand: 20 - Capacity: base_capacity,1.0
1,Mumps,"[428797566.4741963, 313043674.88704294, 375415...",antigen,1,Demand: 20 - Capacity: base_capacity,1.0
2,Rubella,"[397534979.30274767, 280229301.15465266, 34363...",antigen,1,Demand: 20 - Capacity: base_capacity,1.0
3,Diphtheria,"[684660182.764577, 763520991.9887094, 77204363...",antigen,1,Demand: 20 - Capacity: base_capacity,1.0
4,Tetanus,"[693276219.6641592, 771914917.4195288, 7804523...",antigen,1,Demand: 20 - Capacity: base_capacity,1.0
5,Pertussis,"[384795290.1038484, 428153042.7523835, 4343773...",antigen,1,Demand: 20 - Capacity: base_capacity,1.0
6,Hepatitis_B,"[71403062.5729035, 82354829.66282272, 87438322...",antigen,1,Demand: 20 - Capacity: base_capacity,1.0
7,Hib,"[626618410.6704644, 717481444.6596136, 7500777...",antigen,1,Demand: 20 - Capacity: base_capacity,1.0
8,Polio,"[626618410.6704644, 717481444.6596136, 7500777...",antigen,1,Demand: 20 - Capacity: base_capacity,1.0
9,HPV,"[22785851.665604666, 23144268.484348096, 33841...",antigen,1,Demand: 20 - Capacity: base_capacity,1.0


# Create scenario pairs

In [54]:
pair_df.set_index("pair_idx")

,unit,values,type,pair,pair_probability
pair_idx,,,,,
1,Measles,"[420097004.28939533, 368299543.3821963, 405492...",antigen,Demand: 1 - Capacity: base_capacity,1.0
1,Mumps,"[357331305.3951636, 260869729.0725358, 3128461...",antigen,Demand: 1 - Capacity: base_capacity,1.0
1,Rubella,"[331279149.4189564, 233524417.62887722, 286364...",antigen,Demand: 1 - Capacity: base_capacity,1.0
1,Diphtheria,"[570550152.3038142, 636267493.3239245, 6433696...",antigen,Demand: 1 - Capacity: base_capacity,1.0
1,Tetanus,"[577730183.053466, 643262431.1829407, 65037699...",antigen,Demand: 1 - Capacity: base_capacity,1.0
1,Pertussis,"[320662741.753207, 356794202.2936529, 36198112...",antigen,Demand: 1 - Capacity: base_capacity,1.0
1,Hepatitis_B,"[59502552.14408625, 68629024.71901894, 7286526...",antigen,Demand: 1 - Capacity: base_capacity,1.0
1,Hib,"[522182008.8920537, 597901203.8830113, 6250648...",antigen,Demand: 1 - Capacity: base_capacity,1.0
1,Polio,"[522182008.8920537, 597901203.8830113, 6250648...",antigen,Demand: 1 - Capacity: base_capacity,1.0


In [13]:
pair_df.loc[1]

unit                                                            Mumps
values              [276893800, 291347309, 386542696, 259074598, 3...
type                                                          antigen
pair_idx                                                            1
pair                              Demand: 1 - Capacity: base_capacity
pair_probability                                                  1.0
Name: 1, dtype: object

In [14]:
pair_df['unit'].unique()

array(['Measles', 'Mumps', 'Rubella', 'Diphtheria', 'Tetanus', 'Pertussis',
       'Hepatitis_B', 'Hib', 'Polio', 'HPV', 'Rotavirus', 'PCV', 'AJ_Vaccines',
       'BB_NCIPD', 'Bharat_Biotech', 'Bilthoven', 'Biological_E',
       'China_National', 'GSK', 'Haffkine_Bio', 'LG_Chem', 'Merck_Sharp',
       'Panacea_Biotec', 'Pfizer', 'PT_Bio', 'Sanofi', 'Serum_Institute'],
      dtype=object)

In [15]:
pair_df

,unit,values,type,pair_idx,pair,pair_probability
0,Measles,"[346093157, 377569075, 407486189, 271512169, 3...",antigen,1,Demand: 1 - Capacity: base_capacity,1.0
1,Mumps,"[276893800, 291347309, 386542696, 259074598, 3...",antigen,1,Demand: 1 - Capacity: base_capacity,1.0
2,Rubella,"[251625699, 267811068, 370337280, 240424169, 3...",antigen,1,Demand: 1 - Capacity: base_capacity,1.0
3,Diphtheria,"[570550152, 619539916, 626455401, 594496915, 5...",antigen,1,Demand: 1 - Capacity: base_capacity,1.0
4,Tetanus,"[577730183, 626350955, 633278479, 601333076, 5...",antigen,1,Demand: 1 - Capacity: base_capacity,1.0
5,Pertussis,"[320662742, 347414024, 352464578, 335977048, 3...",antigen,1,Demand: 1 - Capacity: base_capacity,1.0
6,Hepatitis_B,"[59502552, 66824756, 70949629, 67124341, 66326...",antigen,1,Demand: 1 - Capacity: base_capacity,1.0
7,Hib,"[522182009, 582182282, 608631765, 590689586, 5...",antigen,1,Demand: 1 - Capacity: base_capacity,1.0
8,Polio,"[522182009, 582182282, 608631765, 590689586, 5...",antigen,1,Demand: 1 - Capacity: base_capacity,1.0
9,HPV,"[18988210, 18779835, 27459977, 72849487, 92670...",antigen,1,Demand: 1 - Capacity: base_capacity,1.0


In [16]:
pair_df.groupby('unit').count()

,values,type,pair_idx,pair,pair_probability
unit,,,,,
AJ_Vaccines,1,1,1,1,1
BB_NCIPD,1,1,1,1,1
Bharat_Biotech,1,1,1,1,1
Bilthoven,1,1,1,1,1
Biological_E,1,1,1,1,1
China_National,1,1,1,1,1
Diphtheria,1,1,1,1,1
GSK,1,1,1,1,1
HPV,1,1,1,1,1
